## 1. Install Dependencies

# Text Processing for All YouTube Transcript JSON Files

This notebook processes **all** raw video JSON files in `../data/raw` and writes timestamp-aware chunk files to `../data/processed`.

## Improvements included

- batch processing for **all** `video_*.json` files
- **timestamp-aware chunks**
- chunk schema now includes:
  - `start_time`
  - `end_time`
  - `start_time_str`
  - `end_time_str`
  - `source_segments`
- batch-level processing summary
- skip handling for files with missing or invalid transcripts

The raw JSON structure is compatible with this workflow because each transcript entry includes `text`, `start`, and `duration`.

In [38]:
import sys
print(sys.executable)

import langchain
print(langchain.__version__)

c:\Users\yogan\Documents\JupyterNotebook_ironhack\Bootcamp\week_08_03_final_project\Final-Project---Youtube-Engineering-Chatbot\venv\Scripts\python.exe
0.2.16


In [39]:
import json
import os
import glob
from copy import deepcopy
from datetime import datetime
from statistics import mean
from typing import Any, Dict, List, Tuple

from pathlib import Path
import pandas as pd

# LangChain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from tqdm import tqdm


## Configuration

Update these values only if you want different chunk sizes or output paths.

In [40]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

# Chunking parameters (optimized for RAG)
CHUNK_SIZE = 700  # characters
CHUNK_OVERLAP = 150  # characters
MIN_CHUNK_SIZE = 200  # minimum viable chunk

# Output settings
CHUNK_FILE_PREFIX = "chunks_"
SUMMARY_FILENAME = "processing_summary.json"

print(f"📁 Raw data: {RAW_DIR}")
print(f"📁 Processed data: {PROCESSED_DIR}")
print(f"📏 Chunk size: {CHUNK_SIZE} chars")
print(f"🔄 Chunk overlap: {CHUNK_OVERLAP} chars")

📁 Raw data: ../data/raw
📁 Processed data: ../data/processed
📏 Chunk size: 700 chars
🔄 Chunk overlap: 150 chars


## Helper functions

In [41]:
def format_seconds(seconds: float) -> str:
    """Convert seconds to HH:MM:SS or MM:SS format."""
    seconds = max(0, float(seconds))
    total = int(seconds)
    hours = total // 3600
    minutes = (total % 3600) // 60
    secs = total % 60
    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def clean_text(text: str) -> str:
    """Clean and normalize text."""
    if text is None:
        return ""
    return " ".join(str(text).replace("\n", " ").split()).strip()


def normalize_transcript_segments(transcript: List[Dict]) -> List[Dict]:
    """Normalize transcript segments with timing info."""
    normalized = []
    for idx, item in enumerate(transcript):
        text = clean_text(item.get("text", ""))
        if not text:
            continue
        
        start = float(item.get("start", 0.0) or 0.0)
        duration = float(item.get("duration", 0.0) or 0.0)
        end = start + max(duration, 0.0)
        
        normalized.append({
            "segment_index": idx,
            "text": text,
            "start": start,
            "duration": duration,
            "end": end,
        })
    return normalized


print("✅ Helper functions defined")

✅ Helper functions defined


## 5. Transcript Processing with LangChain

In [42]:
class TranscriptProcessor:
    """Process YouTube transcripts into RAG-ready chunks using LangChain."""
    
    def __init__(self, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
        # Initialize LangChain text splitter (REQUIRED by Carlos)
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        print(f"✅ LangChain RecursiveCharacterTextSplitter initialized")
        print(f"   Chunk size: {chunk_size}, Overlap: {chunk_overlap}")
    
    def process_video(self, video_data: Dict) -> List[Dict]:
        """Process a single video into chunks."""
        video_id = video_data["video_id"]
        metadata = video_data.get("metadata", {})
        transcript = video_data.get("transcript", [])
        
        if not transcript:
            print(f"⚠️  No transcript for {video_id}")
            return []
        
        # Normalize transcript segments
        normalized_segments = normalize_transcript_segments(transcript)
        if not normalized_segments:
            return []
        
        # Create full text from segments
        full_text = " ".join([seg["text"] for seg in normalized_segments])
        
        # Split using LangChain
        text_chunks = self.text_splitter.split_text(full_text)
        
        # Map chunks back to timestamps
        chunks_with_metadata = self._add_timestamps_to_chunks(
            text_chunks, normalized_segments, video_data
        )
        
        return chunks_with_metadata
    
    def _add_timestamps_to_chunks(self, text_chunks: List[str], 
                                   segments: List[Dict], 
                                   video_data: Dict) -> List[Dict]:
        """Add timestamp metadata to each chunk."""
        chunks = []
        metadata = video_data.get("metadata", {})
        video_id = video_data["video_id"]
        
        # Build character position map
        char_to_segment = []
        current_pos = 0
        for seg in segments:
            seg_text = seg["text"]
            for _ in range(len(seg_text) + 1):  # +1 for space
                char_to_segment.append(seg)
            current_pos += len(seg_text) + 1
        
        # Process each chunk
        current_char = 0
        for idx, chunk_text in enumerate(text_chunks):
            # Find segments that overlap with this chunk
            chunk_start_char = current_char
            chunk_end_char = current_char + len(chunk_text)
            
            # Get relevant segments
            relevant_segments = set()
            for pos in range(chunk_start_char, min(chunk_end_char, len(char_to_segment))):
                relevant_segments.add(char_to_segment[pos]["segment_index"])
            
            if not relevant_segments:
                relevant_segments = {0}
            
            # Get time bounds
            relevant_seg_objs = [segments[i] for i in relevant_segments if i < len(segments)]
            start_time = min([s["start"] for s in relevant_seg_objs]) if relevant_seg_objs else 0
            end_time = max([s["end"] for s in relevant_seg_objs]) if relevant_seg_objs else 0
            
            # Build chunk metadata
            chunk_id = f"{video_id}_chunk_{idx:04d}"
            
            chunk = {
                "chunk_id": chunk_id,
                "video_id": video_id,
                "chunk_index": idx,
                "text": chunk_text,
                "char_count": len(chunk_text),
                "word_count": len(chunk_text.split()),
                "start_time": round(start_time, 2),
                "end_time": round(end_time, 2),
                "start_time_str": format_seconds(start_time),
                "end_time_str": format_seconds(end_time),
                "duration_seconds": round(end_time - start_time, 2),
                "source_segments": sorted(list(relevant_segments)),
                "metadata": {
                    "video_title": metadata.get("title"),
                    "video_url": metadata.get("url"),
                    "channel": metadata.get("channel"),
                    "upload_date": metadata.get("upload_date"),
                    "video_duration": metadata.get("duration"),
                    "view_count": metadata.get("view_count"),
                    "transcript_language": video_data.get("transcript_language"),
                    "description": (metadata.get("description") or "")[:240],
                    "tags": metadata.get("tags") or [],
                    "categories": metadata.get("categories") or [],
                },
            }
            
            chunks.append(chunk)
            current_char = chunk_end_char - CHUNK_OVERLAP  # Account for overlap
        
        # Update total_chunks for each chunk
        for chunk in chunks:
            chunk["total_chunks"] = len(chunks)
        
        return chunks


print("✅ TranscriptProcessor class defined")

✅ TranscriptProcessor class defined


## Discover raw Video JSON files

In [43]:
# Find all raw video JSON files
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "video_*.json")))

print(f"\n📁 Found {len(raw_files)} raw video files")
if raw_files:
    print("\nFirst 5 files:")
    for f in raw_files[:5]:
        print(f"   {os.path.basename(f)}")
else:
    print("⚠️  No raw files found. Run Notebook 01 first!")


📁 Found 65 raw video files

First 5 files:
   video_-CGxZ-qn4HA.json
   video_04K0bLwCDdM.json
   video_0VNIEfX0m4A.json
   video_1YTKedLQOa0.json
   video_21G7LA2DcGQ.json


In [44]:
# Removes duplicates if any (should not be the case, but just in case)
all_json_files = sorted(set(glob.glob(os.path.join(RAW_DIR, "video_*.json"))))

## Process all raw files

This loop reads every `video_*.json` file from `../data/raw`, creates timestamp-aware chunks, and writes one processed JSON per video.

In [45]:
# Initialize processor
processor = TranscriptProcessor(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

# Process all videos
processing_results = []
all_chunks_count = 0

print(f"\n🔄 Processing {len(raw_files)} videos...\n")

for raw_file in tqdm(raw_files, desc="Processing videos"):
    # Load video data
    with open(raw_file, "r", encoding="utf-8") as f:
        video_data = json.load(f)
    
    video_id = video_data["video_id"]
    
    # Process into chunks
    try:
        chunks = processor.process_video(video_data)
        
        if chunks:
            # Save chunks for this video
            output_file = os.path.join(PROCESSED_DIR, f"{CHUNK_FILE_PREFIX}{video_id}.json")
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(chunks, f, indent=2, ensure_ascii=False)
            
            processing_results.append({
                "video_id": video_id,
                "title": video_data["metadata"].get("title"),
                "chunks_created": len(chunks),
                "status": "success",
                "output_file": output_file,
            })
            all_chunks_count += len(chunks)
        else:
            processing_results.append({
                "video_id": video_id,
                "title": video_data["metadata"].get("title"),
                "chunks_created": 0,
                "status": "no_transcript",
                "output_file": None,
            })
    
    except Exception as e:
        print(f"\n❌ Error processing {video_id}: {e}")
        processing_results.append({
            "video_id": video_id,
            "title": video_data["metadata"].get("title"),
            "chunks_created": 0,
            "status": f"error: {str(e)}",
            "output_file": None,
        })

print(f"\n✅ Processing complete!")
print(f"   Videos processed: {len(processing_results)}")
print(f"   Total chunks created: {all_chunks_count}")
print(f"   Avg chunks per video: {all_chunks_count / len(processing_results):.1f}")

✅ LangChain RecursiveCharacterTextSplitter initialized
   Chunk size: 700, Overlap: 150

🔄 Processing 65 videos...



Processing videos: 100%|██████████| 65/65 [00:00<00:00, 163.36it/s]


✅ Processing complete!
   Videos processed: 65
   Total chunks created: 1475
   Avg chunks per video: 22.7


## 8. Save Processing Summary

In [46]:
summary = {
    "processed_at": datetime.now().isoformat(),
    "total_videos": len(processing_results),
    "total_chunks": all_chunks_count,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "text_splitter": "LangChain RecursiveCharacterTextSplitter",
    "videos": processing_results,
}

summary_path = os.path.join(PROCESSED_DIR, SUMMARY_FILENAME)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"📊 Summary saved to: {summary_path}")

📊 Summary saved to: ../data/processed\processing_summary.json


## 9. View Processing Statistics

In [47]:
df_results = pd.DataFrame(processing_results)

print("\n📊 Processing Statistics:")
print(f"   Total videos: {len(df_results)}")
print(f"   Successful: {(df_results['status'] == 'success').sum()}")
print(f"   No transcript: {(df_results['status'] == 'no_transcript').sum()}")
print(f"   Errors: {df_results['status'].str.contains('error').sum()}")
print(f"\n   Total chunks: {df_results['chunks_created'].sum()}")
print(f"   Avg chunks/video: {df_results['chunks_created'].mean():.1f}")
print(f"   Max chunks: {df_results['chunks_created'].max()}")
print(f"   Min chunks: {df_results['chunks_created'].min()}")

print("\n")
display(df_results)


📊 Processing Statistics:
   Total videos: 65
   Successful: 65
   No transcript: 0
   Errors: 0

   Total chunks: 1475
   Avg chunks/video: 22.7
   Max chunks: 40
   Min chunks: 4




,video_id,title,chunks_created,status,output_file
0,-CGxZ-qn4HA,Weighted integral & Weak formulation,25,success,../data/processed\chunks_-CGxZ-qn4HA.json
1,04K0bLwCDdM,The Incredible Properties of Composite Materials,37,success,../data/processed\chunks_04K0bLwCDdM.json
2,0VNIEfX0m4A,"Nodes, Elements & Shape Functions",27,success,../data/processed\chunks_0VNIEfX0m4A.json
3,1YTKedLQOa0,Understanding Torsion,15,success,../data/processed\chunks_1YTKedLQOa0.json
4,21G7LA2DcGQ,Understanding Buckling,23,success,../data/processed\chunks_21G7LA2DcGQ.json
...,...,...,...,...,...
60,to9dVxd-jdE,Different types of Weighted Residual Methods -...,16,success,../data/processed\chunks_to9dVxd-jdE.json
61,tuOlM3P7ygA,Understanding Poisson's Ratio,16,success,../data/processed\chunks_tuOlM3P7ygA.json
62,uaWfZTLNPmA,Understanding Pressure Vessels,17,success,../data/processed\chunks_uaWfZTLNPmA.json
63,xkbQnBAOFEg,"Understanding Failure Theories (Tresca, von Mi...",25,success,../data/processed\chunks_xkbQnBAOFEg.json


## 10. Quality Check: Sample Chunks

In [48]:
# Load a sample chunk file to inspect
chunk_files = glob.glob(os.path.join(PROCESSED_DIR, f"{CHUNK_FILE_PREFIX}*.json"))

if chunk_files:
    sample_file = chunk_files[0]
    print(f"📄 Sample file: {os.path.basename(sample_file)}\n")
    
    with open(sample_file, "r", encoding="utf-8") as f:
        sample_chunks = json.load(f)
    
    print(f"Chunks in file: {len(sample_chunks)}\n")
    print("First chunk structure:")
    print(json.dumps(sample_chunks[0], indent=2)[:1000] + "...")
else:
    print("⚠️  No chunk files found")

📄 Sample file: chunks_-CGxZ-qn4HA.json

Chunks in file: 25

First chunk structure:
{
  "chunk_id": "-CGxZ-qn4HA_chunk_0000",
  "video_id": "-CGxZ-qn4HA",
  "chunk_index": 0,
  "text": "Hello, welcome to basics of finite element analysis, in the last class we were discussing the concept of variation and we were, discussed at length the variational operator so what we will do in the reaming part of this week is we will use that particular concept in context of different types of formulations and starting today we will be discussing weighted integral and weak formulations. So we will illustrate what I intend to talk today and then later generalize through an example for starters so what we will do is let us look",
  "char_count": 533,
  "word_count": 94,
  "start_time": 13.7,
  "end_time": 82.68,
  "start_time_str": "00:13",
  "end_time_str": "01:22",
  "duration_seconds": 68.98,
  "source_segments": [
    0,
    1,
    2,
    3,
    4,
    5,
    6
  ],
  "metadata": {
    "video_title":

## 11. Corpus Quality Assessment

In [49]:
# Load all chunks for quality assessment
all_chunks = []
for chunk_file in chunk_files:
    with open(chunk_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)
        all_chunks.extend(chunks)

# Calculate statistics
char_counts = [c["char_count"] for c in all_chunks]
word_counts = [c["word_count"] for c in all_chunks]
durations = [c["duration_seconds"] for c in all_chunks]

import numpy as np

print("\n" + "="*80)
print("📊 CORPUS QUALITY ASSESSMENT")
print("="*80)

print(f"\n📈 Dataset Metrics:")
print(f"   Total chunks: {len(all_chunks):,}")
print(f"   Total words: {sum(word_counts):,}")
print(f"   Total videos: {len(chunk_files)}")

print(f"\n📏 Chunk Size Distribution:")
print(f"   Characters - Mean: {np.mean(char_counts):.0f}, Median: {np.median(char_counts):.0f}")
print(f"   Characters - Min: {np.min(char_counts)}, Max: {np.max(char_counts)}")
print(f"   Words - Mean: {np.mean(word_counts):.0f}, Median: {np.median(word_counts):.0f}")

print(f"\n⏱️  Duration Distribution:")
print(f"   Mean: {np.mean(durations):.1f}s, Median: {np.median(durations):.1f}s")
print(f"   Min: {np.min(durations):.1f}s, Max: {np.max(durations):.1f}s")

# Quality assessment
benchmarks = {
    "min_chunks": 50,
    "good_chunks": 100,
    "excellent_chunks": 200,
    "min_words": 10000,
    "good_words": 20000,
    "excellent_words": 40000,
}

total_chunks = len(all_chunks)
total_words = sum(word_counts)

print(f"\n✅ Quality Assessment:")
if total_chunks >= benchmarks["excellent_chunks"]:
    print(f"   Chunks: ✅ EXCELLENT ({total_chunks} >= {benchmarks['excellent_chunks']})")
elif total_chunks >= benchmarks["good_chunks"]:
    print(f"   Chunks: 🟢 GOOD ({total_chunks} >= {benchmarks['good_chunks']})")
elif total_chunks >= benchmarks["min_chunks"]:
    print(f"   Chunks: 🟡 ACCEPTABLE ({total_chunks} >= {benchmarks['min_chunks']})")
else:
    print(f"   Chunks: ⚠️  NEEDS MORE ({total_chunks} < {benchmarks['min_chunks']})")

if total_words >= benchmarks["excellent_words"]:
    print(f"   Words: ✅ EXCELLENT ({total_words:,} >= {benchmarks['excellent_words']:,})")
elif total_words >= benchmarks["good_words"]:
    print(f"   Words: 🟢 GOOD ({total_words:,} >= {benchmarks['good_words']:,})")
elif total_words >= benchmarks["min_words"]:
    print(f"   Words: 🟡 ACCEPTABLE ({total_words:,} >= {benchmarks['min_words']:,})")
else:
    print(f"   Words: ⚠️  NEEDS MORE ({total_words:,} < {benchmarks['min_words']:,})")

print("\n" + "="*80)
if total_chunks >= benchmarks["min_chunks"] and total_words >= benchmarks["min_words"]:
    print("✅ READY FOR VECTOR DATABASE INGESTION")
else:
    print("⚠️  CONSIDER ADDING MORE VIDEOS")
print("="*80)


📊 CORPUS QUALITY ASSESSMENT

📈 Dataset Metrics:
   Total chunks: 1,475
   Total words: 159,894
   Total videos: 65

📏 Chunk Size Distribution:
   Characters - Mean: 594, Median: 622
   Characters - Min: 96, Max: 700
   Words - Mean: 108, Median: 110

⏱️  Duration Distribution:
   Mean: 60.3s, Median: 53.4s
   Min: 10.6s, Max: 174.6s

✅ Quality Assessment:
   Chunks: ✅ EXCELLENT (1475 >= 200)
   Words: ✅ EXCELLENT (159,894 >= 40,000)

✅ READY FOR VECTOR DATABASE INGESTION
